# Cross-filter Opportunity Atlas — Notebook 03

Descriptive analysis of a **pinned canonical bundle**, presented through a verified atlas.
This notebook does not discover validated edges, recommend filter changes, or place wagers.

**Population:** logged Bet Tracker records, not the full available market. Opportunity IDs
are proxies; source timezone, chronology, export scope, and fee inclusion remain unverified.
Real-money labels use the historical user assumption; paper trades do not prove executable fills.

Notebook 01 and Notebook 02 retain their historical inputs and metrics.

## Explicit input and verified loading

Build an atlas with `scripts/build_opportunity_atlas.py` first. Set `ODDSJAM_ATLAS_DIR`
to its directory before starting the kernel. There is no implicit latest selection.
If using an existing kernel, restart it after changing its environment.
Private outputs belong under ignored data paths; do not commit executed monetary reports.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from oddsjam_ev.analysis.opportunity_atlas import summarize_wagers
from oddsjam_ev.io.atlas import load_atlas

selected = os.environ.get("ODDSJAM_ATLAS_DIR")
if not selected:
    raise ValueError("Set ODDSJAM_ATLAS_DIR to an explicit published atlas directory.")
atlas_dir = Path(selected).expanduser().resolve()
tables = load_atlas(atlas_dir)
manifest = json.loads((atlas_dir / "manifest.json").read_text())
wagers = tables["wager_dimensions"]
segments = tables["wager_segments"]
proxies = tables["opportunity_segments"]
display(pd.Series(manifest["specification"]["limitations"], name="Interpretation limits"))
print("Atlas:", manifest["atlas_id"])
print("Canonical bundle:", manifest["specification"]["canonical_bundle_id"])

## Population reconciliation and execution status

Only eligible single candidates contribute reported stake/profit and ROI. Tickets,
refunds, unknown products, and other exclusions remain visible in candidate counts.
The three execution modes are separate; unknown does not mean funded.
Opportunity counts across modes and filter memberships are **non-additive**.

In [ ]:
overall = segments.loc[segments["view"].eq("overall")]
assert overall["candidate_count"].sum() == len(wagers)
assert overall["eligible_count"].sum() == wagers["reported_roi_eligible"].sum()
display(overall)
display(proxies.loc[proxies["view"].eq("overall")])

## Feature coverage and market mapping

Mappings use exact market labels; unmapped labels are retained. EV is source-advertised
percentage, not realized ROI or CLV-derived EV. Extreme EV values remain in open-ended
tails. Liquidity is not verified; no capacity or timing analysis is performed.
Conflict categories preserve disagreeing source metadata.

In [ ]:
quality = tables["atlas_quality"]
display(
    quality.loc[quality["category"].isin(["<missing>", "<invalid>", "<conflict>", "<unmapped>"])]
)
display(
    wagers.loc[wagers["market_family"].eq("<unmapped>"), "market_name"]
    .value_counts()
    .rename("candidate_count")
    .to_frame()
)
display(
    quality.loc[quality["dimension"].isin(["canonical_disposition", "economic_exclusions_json"])]
)

## Broad sportsbook, league, and market views

These views describe the observed sample. Positive historical ROI is not a deployable edge.
Rows separate product and execution status; do not pool paper and real results.

In [ ]:
for view in ["sportsbook", "league", "market_family"]:
    print(view)
    display(segments.loc[segments["view"].eq(view)])

volume = overall.groupby("execution_status")["candidate_count"].sum()
if not volume.empty:
    volume.plot.bar(title="Logged wager candidates by assumed execution mode", ylabel="Candidates")
    plt.tight_layout()
    plt.show()

## Odds and advertised-EV bands

Intervals are left closed and right open. Display boundaries are configurable and
not betting recommendations. Missing or invalid values are separate categories.

In [ ]:
for view in ["odds_band", "advertised_ev_band", "odds_band__advertised_ev_band"]:
    display(segments.loc[segments["view"].eq(view)])

## Selected comparisons and explicit drill-down

Choose a descriptive slice without ranking segments as strategies. Tables use each wager
once per slice. Distinct event/selection/proxy counts are dependence diagnostics, not
estimates of effective independent sample size.

In [ ]:
display(segments.loc[segments["view"].eq("sportsbook__market_family")])
display(segments.loc[segments["view"].eq("league__market_family")])
# Change these display choices explicitly; no minimum-sample promotion rule is applied.
execution_mode = "real_money"
selected_dimensions = ["sportsbook", "market_family", "league"]
selected_wagers = wagers.loc[wagers["execution_status"].eq(execution_mode)]
display(summarize_wagers(selected_wagers, selected_dimensions))

## Within-filter diagnostics and overlap

Filter labels are attribution metadata, not discovery keys. A wager can belong to more
than one filter. Never sum these rows into a portfolio total. Unresolved and missing
labels stay visible. Parent/subsegment signs are descriptive, not action triggers.

In [ ]:
filters = tables["filter_segment_diagnostics"]
parent = filters.loc[filters["view"].eq("filter_id__attribution_status")]
children = filters.loc[filters["view"].eq("filter_id__attribution_status__market_family")]
keys = ["execution_status", "product", "filter_id", "attribution_status"]
comparison = children.merge(
    parent[keys + ["reported_roi"]].rename(columns={"reported_roi": "parent_reported_roi"}),
    on=keys,
    validate="many_to_one",
)
display(comparison)
display(filters.loc[filters["attribution_status"].ne("resolved")])

## Concentration and uncertainty

Largest stake share and largest absolute-profit share reveal concentration. Null values
mean there is no eligible denominator. No confidence intervals are supplied: dependence,
chronology, selection bias, and multiple comparisons have not been validated. Formal
robustness, cluster resampling policy, holdouts, and promotion belong to later work.

In [ ]:
display(
    overall[
        [
            "execution_status",
            "product",
            "eligible_count",
            "eligible_event_id_count",
            "eligible_missing_event_id_count",
            "largest_stake_share",
            "largest_absolute_profit_share",
            "uncertainty_status",
        ]
    ]
)

## Research gaps and interpretation

- This is a distribution of logged records, not scanner coverage or fillability.
- Reported profit is preserved; fees are unverified and no estimated fees are deducted.
- All October wager timestamps remain timezone-unknown; no time ranking is justified.
- Explicit parlay tickets and unlinked Pick’em candidates remain outside eligible ROI.
- Filter configurations are dated context, not proof of settings on historical wagers.
- Mapping gaps, unsupported products, and unresolved source records require evidence.
- No segment is promoted, no filters are edited, and no weekly job is enabled.